# Generative Models Introduction

Exploring Autoencoders, VAEs, and GANs using NumPy and scikit-learn on simple data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
np.random.seed(42)
%matplotlib inline

## 1. Simple Autoencoder with sklearn

In [ ]:
# Load and preprocess digits
digits = load_digits()
X = digits.data / 16.0  # normalize to [0, 1]
print(f'Data shape: {X.shape} (each sample is an 8x8 image flattened to 64 dims)')

# Build autoencoder using MLPRegressor (encoder-decoder)
# Architecture: 64 -> 32 -> 2 -> 32 -> 64
# We'll train it to reconstruct its own input
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 2, 32),  # bottleneck of 2 dims
    activation='relu',
    max_iter=500,
    random_state=42,
    learning_rate_init=0.001
)

# Train: input = output = X
autoencoder.fit(X, X)
X_reconstructed = autoencoder.predict(X)

# Compute reconstruction error
mse = np.mean((X - X_reconstructed) ** 2)
print(f'Reconstruction MSE: {mse:.4f}')

In [ ]:
# Visualize original vs reconstructed
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(X_reconstructed[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Reconstructed', fontsize=12)
plt.suptitle('Autoencoder: Original vs Reconstructed', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Visualize Latent Space

In [ ]:
# Extract latent representations using a 2D bottleneck
# We need to manually get the activations at the bottleneck
# Simpler approach: train encoder separately with PCA-like objective

# Use a dedicated encoder: 64 -> 32 -> 2
from sklearn.neural_network import MLPRegressor

# Train autoencoder with 2D latent space
ae_2d = MLPRegressor(hidden_layer_sizes=(32, 2, 32), activation='relu',
                     max_iter=1000, random_state=42, learning_rate_init=0.001)
ae_2d.fit(X, X)

# Extract 2D latent codes by forward propagating through first layers
def get_latent_codes(model, X):
    """Extract activations at the bottleneck layer."""
    a = X
    # Forward through layers until bottleneck (layer index 1 for 3-layer model)
    for i in range(2):  # first 2 weight matrices: input->32, 32->2
        a = a @ model.coefs_[i] + model.intercepts_[i]
        a = np.maximum(0, a)  # relu
    return a

latent_codes = get_latent_codes(ae_2d, X)
print(f'Latent codes shape: {latent_codes.shape}')

plt.figure(figsize=(10, 8))
scatter = plt.scatter(latent_codes[:, 0], latent_codes[:, 1], 
                      c=digits.target, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.xlabel('Latent Dim 1')
plt.ylabel('Latent Dim 2')
plt.title('Autoencoder 2D Latent Space')
plt.show()

## 3. Basic GAN Concept (1D Toy Distribution)

In [ ]:
# GAN on a toy 1D problem: learn to generate from a Gaussian mixture
# Real distribution: mixture of two Gaussians
def sample_real(n):
    """Sample from true distribution (mixture of 2 Gaussians)."""
    choices = np.random.choice([0, 1], size=n)
    samples = np.where(choices == 0, 
                       np.random.normal(-2, 0.5, n),
                       np.random.normal(2, 0.5, n))
    return samples.reshape(-1, 1)

def sample_noise(n, dim=1):
    """Sample noise for generator input."""
    return np.random.randn(n, dim)

# Simple neural network functions
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

class SimpleGenerator:
    def __init__(self, input_dim=1, hidden_dim=32, output_dim=1):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros(output_dim)
    
    def forward(self, z):
        self.z = z
        self.h = relu(z @ self.W1 + self.b1)
        return self.h @ self.W2 + self.b2

class SimpleDiscriminator:
    def __init__(self, input_dim=1, hidden_dim=32):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, 1) * 0.1
        self.b2 = np.zeros(1)
    
    def forward(self, x):
        self.x = x
        self.h = relu(x @ self.W1 + self.b1)
        logits = self.h @ self.W2 + self.b2
        return sigmoid(logits)

print('Generator: noise -> hidden -> fake samples')
print('Discriminator: samples -> hidden -> real/fake probability')

In [ ]:
# Train GAN (simplified training loop)
G = SimpleGenerator(1, 64, 1)
D = SimpleDiscriminator(1, 64)
lr_g = 0.01
lr_d = 0.01

d_losses = []
g_losses = []
gen_samples_history = []

for epoch in range(2000):
    batch_size = 128
    
    # --- Train Discriminator ---
    real_data = sample_real(batch_size)
    noise = sample_noise(batch_size)
    fake_data = G.forward(noise)
    
    # D scores
    d_real = D.forward(real_data)
    d_fake = D.forward(fake_data)
    
    # D loss: -[log(D(real)) + log(1-D(fake))]
    d_loss = -np.mean(np.log(d_real + 1e-8) + np.log(1 - d_fake + 1e-8))
    
    # Simple gradient update for D (approximate with finite differences)
    eps = 1e-4
    for param_name in ['W1', 'b1', 'W2', 'b2']:
        param = getattr(D, param_name)
        grad = np.zeros_like(param)
        flat_param = param.ravel()
        # Stochastic coordinate descent (update random subset)
        indices = np.random.choice(len(flat_param), min(10, len(flat_param)), replace=False)
        for idx in indices:
            flat_param[idx] += eps
            d_real_p = D.forward(real_data)
            d_fake_p = D.forward(fake_data)
            loss_p = -np.mean(np.log(d_real_p + 1e-8) + np.log(1 - d_fake_p + 1e-8))
            flat_param[idx] -= 2*eps
            d_real_m = D.forward(real_data)
            d_fake_m = D.forward(fake_data)
            loss_m = -np.mean(np.log(d_real_m + 1e-8) + np.log(1 - d_fake_m + 1e-8))
            flat_param[idx] += eps  # restore
            grad.ravel()[idx] = (loss_p - loss_m) / (2*eps)
        setattr(D, param_name, param - lr_d * grad)
    
    # --- Train Generator ---
    noise = sample_noise(batch_size)
    fake_data = G.forward(noise)
    d_fake = D.forward(fake_data)
    g_loss = -np.mean(np.log(d_fake + 1e-8))
    
    for param_name in ['W1', 'b1', 'W2', 'b2']:
        param = getattr(G, param_name)
        grad = np.zeros_like(param)
        flat_param = param.ravel()
        indices = np.random.choice(len(flat_param), min(10, len(flat_param)), replace=False)
        for idx in indices:
            flat_param[idx] += eps
            fake_p = G.forward(noise)
            loss_p = -np.mean(np.log(D.forward(fake_p) + 1e-8))
            flat_param[idx] -= 2*eps
            fake_m = G.forward(noise)
            loss_m = -np.mean(np.log(D.forward(fake_m) + 1e-8))
            flat_param[idx] += eps
            grad.ravel()[idx] = (loss_p - loss_m) / (2*eps)
        setattr(G, param_name, param - lr_g * grad)
    
    d_losses.append(d_loss)
    g_losses.append(g_loss)
    
    if epoch % 500 == 0:
        gen_samples_history.append(G.forward(sample_noise(1000)))
        print(f'Epoch {epoch}: D_loss={d_loss:.4f}, G_loss={g_loss:.4f}')

## 4. GAN Training Dynamics

In [ ]:
# Plot training dynamics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(d_losses[::10], label='Discriminator Loss', alpha=0.7)
axes[0].plot(g_losses[::10], label='Generator Loss', alpha=0.7)
axes[0].set_xlabel('Epoch (x10)')
axes[0].set_ylabel('Loss')
axes[0].set_title('GAN Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Generated distribution over time
real = sample_real(2000)
axes[1].hist(real, bins=50, alpha=0.5, density=True, label='Real', color='blue')
final_fake = G.forward(sample_noise(2000))
axes[1].hist(final_fake, bins=50, alpha=0.5, density=True, label='Generated', color='red')
axes[1].set_title('Real vs Generated Distribution')
axes[1].legend()
axes[1].set_xlabel('Value')

plt.tight_layout()
plt.show()

## 5. Mode Collapse Demonstration

In [ ]:
# Mode collapse: when generator only produces one mode
# Simulate by showing a poorly trained generator
print('Mode Collapse: Generator learns to produce only ONE mode of the distribution')
print('instead of covering all modes.\n')

# Simulate mode collapse
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Real distribution (3 modes)
real_3mode = np.concatenate([np.random.normal(-4, 0.5, 500),
                             np.random.normal(0, 0.5, 500),
                             np.random.normal(4, 0.5, 500)])

# Good generator (covers all modes)
good_gen = np.concatenate([np.random.normal(-3.8, 0.6, 500),
                           np.random.normal(0.1, 0.6, 500),
                           np.random.normal(3.9, 0.6, 500)])

# Mode-collapsed generator (only one mode)
collapsed_gen = np.random.normal(0, 0.8, 1500)

for ax, gen, title in zip(axes, [real_3mode, good_gen, collapsed_gen],
                          ['Real Distribution (3 modes)', 'Good Generator', 'Mode Collapse']):
    ax.hist(gen, bins=60, density=True, alpha=0.7, color='steelblue')
    ax.hist(real_3mode, bins=60, density=True, alpha=0.3, color='red', label='Real')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()
print('Mode collapse is a major GAN training failure mode.')
print('Solutions: Wasserstein loss, minibatch discrimination, unrolled GANs')

## 6. Variational Autoencoder (VAE) Concept

In [ ]:
# VAE key idea: encode to a DISTRIBUTION, not a point
# Then sample from that distribution (reparameterization trick)

print('=== Autoencoder vs VAE ===')
print()
print('Autoencoder:')
print('  Input -> Encoder -> z (point) -> Decoder -> Reconstruction')
print()
print('VAE:')
print('  Input -> Encoder -> (mu, log_var) -> z = mu + exp(0.5*log_var) * epsilon -> Decoder')
print()
print('Reparameterization Trick:')
print('  z = mu + sigma * epsilon,  where epsilon ~ N(0,1)')
print('  This allows gradients to flow through the sampling step!')

# Implement VAE components
class SimpleVAE:
    """Simplified VAE using numpy (forward pass only for demonstration)."""
    
    def __init__(self, input_dim=64, hidden_dim=32, latent_dim=2):
        # Encoder weights
        self.W_enc = np.random.randn(input_dim, hidden_dim) * 0.01
        self.b_enc = np.zeros(hidden_dim)
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * 0.01
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * 0.01
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder weights
        self.W_dec1 = np.random.randn(latent_dim, hidden_dim) * 0.01
        self.b_dec1 = np.zeros(hidden_dim)
        self.W_dec2 = np.random.randn(hidden_dim, input_dim) * 0.01
        self.b_dec2 = np.zeros(input_dim)
    
    def encode(self, x):
        h = np.maximum(0, x @ self.W_enc + self.b_enc)
        mu = h @ self.W_mu + self.b_mu
        log_var = h @ self.W_logvar + self.b_logvar
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """The reparameterization trick!"""
        std = np.exp(0.5 * log_var)
        epsilon = np.random.randn(*mu.shape)  # sample from N(0,1)
        z = mu + std * epsilon
        return z
    
    def decode(self, z):
        h = np.maximum(0, z @ self.W_dec1 + self.b_dec1)
        x_recon = 1 / (1 + np.exp(-(h @ self.W_dec2 + self.b_dec2)))  # sigmoid
        return x_recon
    
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decode(z)
        return x_recon, mu, log_var, z

vae = SimpleVAE(input_dim=64, hidden_dim=32, latent_dim=2)
print('\nVAE initialized with 2D latent space')

In [ ]:
# VAE Loss: Reconstruction + KL Divergence
def vae_loss(x, x_recon, mu, log_var):
    # Reconstruction loss (binary cross-entropy)
    recon_loss = -np.mean(np.sum(
        x * np.log(x_recon + 1e-8) + (1-x) * np.log(1 - x_recon + 1e-8), axis=1))
    
    # KL divergence: pushes latent distribution toward N(0,1)
    kl_loss = -0.5 * np.mean(np.sum(1 + log_var - mu**2 - np.exp(log_var), axis=1))
    
    return recon_loss, kl_loss, recon_loss + kl_loss

# Demonstrate on a batch
batch = X[:32]
x_recon, mu, log_var, z = vae.forward(batch)
recon_l, kl_l, total_l = vae_loss(batch, x_recon, mu, log_var)

print(f'Reconstruction Loss: {recon_l:.2f}')
print(f'KL Divergence Loss:  {kl_l:.2f}')
print(f'Total VAE Loss:      {total_l:.2f}')
print(f'\nKL divergence encourages the latent space to be smooth and continuous')

## 7. Compare Reconstructions: AE vs VAE

In [ ]:
# Train a proper autoencoder with sklearn and compare
# AE: deterministic latent space
ae = MLPRegressor(hidden_layer_sizes=(32, 2, 32), activation='relu',
                  max_iter=1000, random_state=42, learning_rate_init=0.001)
ae.fit(X, X)
X_ae_recon = ae.predict(X)

# For VAE, use the untrained forward pass (conceptual comparison)
X_vae_recon, _, _, _ = vae.forward(X)

fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for i in range(10):
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(np.clip(X_ae_recon[i], 0, 1).reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(np.clip(X_vae_recon[i], 0, 1).reshape(8, 8), cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('AE (trained)', fontsize=11)
axes[2, 0].set_ylabel('VAE (untrained)', fontsize=11)
plt.suptitle('AE (trained) produces good reconstructions; VAE needs training too', fontsize=12)
plt.tight_layout()
plt.show()

print('Key difference: VAE latent space is regularized (smooth, continuous)')
print('This allows meaningful interpolation and generation from the latent space')

## 8. Generate New Samples from Latent Space

In [ ]:
# Generate new digits by sampling the trained AE's latent space
# Get latent codes from trained AE
def get_ae_latent(model, X):
    a = X
    for i in range(2):  # through first 2 layers to bottleneck
        a = a @ model.coefs_[i] + model.intercepts_[i]
        a = np.maximum(0, a)
    return a

def decode_from_latent(model, z):
    """Decode from 2D latent space through remaining layers."""
    a = z
    for i in range(2, len(model.coefs_)):
        a = a @ model.coefs_[i] + model.intercepts_[i]
        if i < len(model.coefs_) - 1:
            a = np.maximum(0, a)  # relu for hidden, linear for output
    return a

latent = get_ae_latent(ae, X)
print(f'Latent space range: x=[{latent[:,0].min():.1f}, {latent[:,0].max():.1f}], '
      f'y=[{latent[:,1].min():.1f}, {latent[:,1].max():.1f}]')

# Generate by sampling grid in latent space
n_grid = 10
x_range = np.linspace(latent[:,0].min(), latent[:,0].max(), n_grid)
y_range = np.linspace(latent[:,1].min(), latent[:,1].max(), n_grid)

fig, axes = plt.subplots(n_grid, n_grid, figsize=(12, 12))
for i, y_val in enumerate(reversed(y_range)):
    for j, x_val in enumerate(x_range):
        z = np.array([[x_val, y_val]])
        generated = decode_from_latent(ae, z)
        axes[i, j].imshow(np.clip(generated.reshape(8, 8), 0, 1), cmap='gray')
        axes[i, j].axis('off')

plt.suptitle('Generated Digits: Sampling from Latent Space Grid', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print('Smooth transitions in latent space = smooth transitions in generated images')

In [ ]:
# Interpolation between two digits
idx1, idx2 = 0, 100  # digit 0 and digit 1
z1 = get_ae_latent(ae, X[idx1:idx1+1])
z2 = get_ae_latent(ae, X[idx2:idx2+1])

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, alpha in enumerate(np.linspace(0, 1, 10)):
    z_interp = (1 - alpha) * z1 + alpha * z2
    generated = decode_from_latent(ae, z_interp)
    axes[i].imshow(np.clip(generated.reshape(8, 8), 0, 1), cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'{alpha:.1f}')

plt.suptitle(f'Latent Space Interpolation: digit {digits.target[idx1]} -> digit {digits.target[idx2]}', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Model | Key Idea | Latent Space | Generation Quality |
|-------|----------|--------------|-------------------|
| Autoencoder | Compress & reconstruct | Unstructured | Poor (gaps in latent space) |
| VAE | Regularized latent space | Smooth, continuous | Better (can sample anywhere) |
| GAN | Adversarial training | Implicit | Best (but hard to train) |

**Key Concepts:**
- AE: minimize reconstruction error
- VAE: minimize reconstruction + KL divergence (regularization)
- GAN: minimax game between generator and discriminator